In [ ]:
import pandas as pd
from pathlib import Path
from data_profiling import ProfileReport
import ipywidgets as widgets
from IPython.display import display


# Caminhos
DATA_PATH = Path("../data/raw")
REPORT_PATH = Path("../reports")

# Garante que reports existe
REPORT_PATH.mkdir(exist_ok=True)


# Lista CSVs
datasets = {
    file.name: file 
    for file in DATA_PATH.glob("*.csv")
}


# Menu
menu = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Base:",
    layout=widgets.Layout(width="400px")
)


botao = widgets.Button(
    description="Gerar Relatório",
    button_style="success"
)


saida = widgets.Output()


def gerar_relatorio(b):

    with saida:
        saida.clear_output()

        arquivo = datasets[menu.value]

        print(f"Analisando: {arquivo.name}")

        df = pd.read_csv(arquivo)

        print(f"Linhas: {df.shape[0]}")
        print(f"Colunas: {df.shape[1]}")


        profile = ProfileReport(
            df,
            title=f"FG Profiling - {menu.value}",
            explorative=True
        )


        # Salvar relatório
        nome_relatorio = menu.value.replace(".csv", "_profiling.html")

        caminho_saida = REPORT_PATH / nome_relatorio

        profile.to_file(caminho_saida)

        print(f"\nRelatório salvo em:")
        print(caminho_saida)


        display(profile)


botao.on_click(gerar_relatorio)


display(menu)
display(botao)
display(saida)

Dropdown(description='Base:', layout=Layout(width='400px'), options=('olist_customers_dataset.csv', 'olist_geo…

Button(button_style='success', description='Gerar Relatório', style=ButtonStyle())

Output()

In [6]:
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
import ipywidgets as widgets
from IPython.display import display


# Caminhos
DATA_PATH = Path("../data/raw")


# Lista CSVs
datasets = {
    file.name: file
    for file in DATA_PATH.glob("*.csv")
}


# Menu
menu = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Base:",
    layout=widgets.Layout(width="400px")
)


botao = widgets.Button(
    description="Validar Base",
    button_style="success"
)


saida = widgets.Output()


def validar_base(b):

    with saida:
        saida.clear_output()

        arquivo = datasets[menu.value]

        print(f"Validando: {arquivo.name}")

        df = pd.read_csv(arquivo)

        print(f"Linhas: {df.shape[0]}")
        print(f"Colunas: {df.shape[1]}")
        print()


        # Infere um schema inicial a partir do dataframe
        schema = pa.infer_schema(df)


        print("SCHEMA INFERIDO")
        print("-" * 50)

        for coluna, regra in schema.columns.items():

            print(
                f"{coluna:<40} "
                f"dtype={str(regra.dtype):<15} "
                f"nullable={regra.nullable}"
            )


        print("\nVALIDAÇÃO")
        print("-" * 50)


        try:

            schema.validate(
                df,
                lazy=True
            )

            print("✅ Base validada com sucesso.")
            print("Nenhum erro encontrado pelo schema.")


        except pa.errors.SchemaErrors as erro:

            print("❌ Foram encontrados problemas na base.\n")

            erros = erro.failure_cases

            display(erros)


botao.on_click(validar_base)


display(menu)
display(botao)
display(saida)

Dropdown(description='Base:', layout=Layout(width='400px'), options=('olist_customers_dataset.csv', 'olist_geo…

Button(button_style='success', description='Validar Base', style=ButtonStyle())

Output()